# Median |error| by residue type × secondary structure × exposure (PANAV test set, 10-seed ensemble)

Residues: GT present, prediction mask True, no proline.

In [ ]:
import os
os.makedirs("Figs", exist_ok=True)  # figures are written to ./Figs

import json
from pathlib import Path
import numpy as np
import pandas as pd

SS_ORDER       = ['helix', 'sheet', 'coil']
EXPOSURE_ORDER = ['buried', 'exposed']

PRED = '../../data/analysis/Fig4/results/predictions_N_seed_ensemble_10seed.json'
GT   = '../../data/rBMRB/cluster_30p/PANAV/test.json'
PRR  = '../../data/analysis/Fig4/sasa_results/per_residue_records.json'

def set_style(base=12, family='Arial'):
    import matplotlib.pyplot as plt
    plt.rcParams.update({
        'font.family': family, 'font.size': base,
        'axes.titlesize': base + 2, 'axes.labelsize': base,
        'xtick.labelsize': base - 2, 'ytick.labelsize': base - 2,
        'legend.fontsize': base - 2, 'svg.fonttype': 'none',
    })

## 1. Load predictions, ground truth, and per_residue_records (ss_class + exposure)

In [ ]:
with open(PRED) as f: pred_data = json.load(f)
with open(GT)   as f: gt_data   = json.load(f)
with open(PRR)  as f: prr_data  = json.load(f)

# 10-seed ensemble: use the precomputed per-residue "mean" (with "mask"), not the per-seed "pred" matrix
pred_dict, mask_dict = pred_data['mean'], pred_data['mask']
dictionary = {v: k for k, v in gt_data['entry_ID'].items()}
print('Predictions        :', len(pred_dict), 'entries')
print('per_residue_records:', len(prr_data), 'entries  | fields:', list(next(iter(prr_data.values()))[0].keys()))

## 2. Merge: (residue_type, ss_class, exposure, |error|) per residue

In [ ]:
records = []
length_mismatch = 0
missing_prr = 0
for entry in gt_data['entry_ID'].values():
    key = dictionary.get(entry)
    if key is None or entry not in pred_dict:
        continue

    gt_seq, gt_shifts = gt_data['sequence'][key], gt_data['N'][key]
    pred_arr_full = np.array(pred_dict[entry])
    pred_mask_full = np.array(mask_dict[entry], dtype=bool)
    n = len(gt_seq)
    # Prediction arrays are padded to the longest sequence in the batch; any tail beyond gt_seq is
    # mask=False / mean=None padding, so truncate to n instead of dropping the protein
    if len(pred_arr_full) < n or len(pred_mask_full) < n:
        length_mismatch += 1
        continue
    pred_arr = pred_arr_full[:n]
    pred_mask = pred_mask_full[:n]

    prr_entry = prr_data.get(entry)
    if prr_entry is None:
        continue
    prr_lookup = {r['resnum']: r for r in prr_entry}   # resnum -> {ss_class, exposure, ...}

    # Proline / None exclusion + the prediction's own mask (tag/flag and outlier exclusion are already in it;
    # the GT N_mask/flag and the 4-sigma BMRB filter are not applied again)
    gt_mask_1 = np.array([res != 'P' for res in gt_seq])
    none_mask = np.array([g is not None for g in gt_shifts])
    mask = gt_mask_1 & none_mask & pred_mask
    if mask.sum() == 0:
        continue

    full_indices = np.where(mask)[0]
    pred_f = pred_arr[mask]
    gt_f   = np.array(gt_shifts)[mask].astype(float)
    abs_error = np.abs(pred_f - gt_f)   # no analytical centering (err - err.mean())

    for i, full_idx in enumerate(full_indices):
        resnum = int(full_idx) + 1
        rec    = prr_lookup.get(resnum)
        if rec is None or rec.get('ss_class') is None or rec.get('exposure') not in ('buried', 'exposed'):
            missing_prr += 1
            continue
        records.append({
            'entry':        entry,
            'residue_type': gt_seq[full_idx],
            'ss_class':     rec['ss_class'],
            'exposure':     rec['exposure'],
            'abs_error':    abs_error[i],
        })

df = pd.DataFrame(records)
print(f'Total residues : {len(df)}  |  length mismatch skipped: {length_mismatch}  |  not in per_residue_records / not buried|exposed: {missing_prr}')
print('\nExposure split:'); print(df['exposure'].value_counts().to_string())
print('\nExposure x SS counts:')
print(df.groupby(['exposure','ss_class']).size().unstack('ss_class').reindex(columns=SS_ORDER).reindex(EXPOSURE_ORDER))

## 3. Pivots: median |error| per (residue × SS) within each exposure

In [ ]:
def pivots(sub):
    med = (sub.groupby(['residue_type', 'ss_class'])['abs_error'].median()
              .unstack('ss_class').reindex(columns=SS_ORDER))
    iqr = (sub.groupby(['residue_type', 'ss_class'])['abs_error']
              .agg(lambda x: np.subtract(*np.percentile(x, [75, 25])))
              .unstack('ss_class').reindex(columns=SS_ORDER))
    cnt = (sub.groupby(['residue_type', 'ss_class'])['abs_error'].count()
              .unstack('ss_class').reindex(columns=SS_ORDER))
    return med, iqr, cnt

# residue order: pooled median |error| across ALL SS & exposure, descending
order = list(df.groupby('residue_type')['abs_error'].median().sort_values(ascending=False).index)

panels = {}
for exp in EXPOSURE_ORDER:
    med, iqr, cnt = pivots(df[df['exposure'] == exp])
    panels[exp] = (med.reindex(order), iqr.reindex(order), cnt.reindex(order))

for exp in EXPOSURE_ORDER:
    print(f'\n=== {exp} : median |error| (ppm) ===')
    print(panels[exp][0].round(3))

In [ ]:
import matplotlib.pyplot as plt
set_style(base=14)

# ---- standard amino-acid classes (P is already excluded) ----
AA_GROUPS = {
    'Aliphatic': ['G', 'A', 'V', 'L', 'I', 'M'],
    'Aromatic':  ['F', 'Y', 'W'],
    'Polar':     ['S', 'T', 'C', 'N', 'Q'],
    'Positive':  ['K', 'R', 'H'],
    'Negative':  ['D', 'E'],
}

# pooled median |error| per residue type (used for ordering)
res_med = df.groupby('residue_type')['abs_error'].median()

# within a class: residue types present, by descending median
group_sorted = {}
for g, aas in AA_GROUPS.items():
    present = sorted([a for a in aas if a in res_med.index],
                     key=lambda a: res_med[a], reverse=True)
    if present:
        group_sorted[g] = present

# between classes: by descending within-class maximum median
groups_ordered = sorted(group_sorted,
                        key=lambda g: max(res_med[a] for a in group_sorted[g]),
                        reverse=True)

new_order = [a for g in groups_ordered for a in group_sorted[g]]
# class boundaries (for separator lines / labels) as cumulative positions in new_order
group_bounds, acc = [], 0
for g in groups_ordered:
    acc += len(group_sorted[g])
    group_bounds.append((g, acc - len(group_sorted[g]), acc))   # (name, start, end), half-open

# reorder every panel
panels = {e: tuple(t.reindex(new_order) for t in panels[e]) for e in EXPOSURE_ORDER}

ss_disp = ['Helix', 'Sheet', 'Coil']
all_med = np.concatenate([panels[e][0].values.ravel() for e in EXPOSURE_ORDER])
vmin = np.floor(np.nanmin(all_med) * 2) / 2
vmax = 1.5          # set from the bulk of the data, e.g. 0-1.5 ppm

thresh = vmin + 0.65 * (vmax - vmin)     # black/white text threshold follows the colour range

n = len(EXPOSURE_ORDER)
fig, axes = plt.subplots(1, n, figsize=(6 * n, 9), sharey=True)
if n == 1: axes = [axes]
im = None
for ax, exp in zip(axes, EXPOSURE_ORDER):
    med, iqr, cnt = panels[exp]
    im = ax.imshow(med.values, aspect='auto', cmap='YlOrRd', vmin=vmin, vmax=vmax)
    ss_n = np.nansum(cnt.values, axis=0).astype(int)   # total per SS column (summed over residue types)
    ax.set_xticks(range(3))
    ax.set_xticklabels([f'{s}\n(n={nc})' for s, nc in zip(ss_disp, ss_n)])
    ax.set_yticks(range(len(med))); ax.set_yticklabels(med.index)
    ax.set_xlabel('Secondary structure')
    n_exp = int(np.nansum(cnt.values))
    ax.set_title(f'{exp.capitalize()}  (n={n_exp})')
    for i in range(len(med)):
        for j in range(3):
            v = med.values[i, j]
            if not np.isnan(v):
                c = int(cnt.values[i, j]) if not np.isnan(cnt.values[i, j]) else 0
                ax.text(j, i, f'{v:.2f} | {iqr.values[i, j]:.2f} | {c:03d}',
                        ha='center', va='center', fontsize=10, font='Arial',
                        color='white' if v > thresh else 'black')
    # ---- class separator lines ----
    for _, start, end in group_bounds[1:]:
        ax.axhline(start - 0.5, color='k', lw=1.2)

# ---- class labels (left of the leftmost panel) ----
for g, start, end in group_bounds:
    axes[0].text(-1, (start + end - 1) / 2, g, ha='center', va='center', fontsize=12, font='Arial', fontweight='bold')

cbar = fig.colorbar(im, ax=axes, fraction=0.046, pad=0.02)
cbar.set_label('Median |error| (ppm, predicted by SLAE-CS)', rotation=270, labelpad=16)
fig.text(0.5, 0.94, 'Cell annotation:  Median | IQR | Count', ha='center', va='top')

# fig.savefig("Figs/residue_ss_exposure_heatmap_panav10seedens.svg", format='svg', bbox_inches='tight')
# fig.savefig("Figs/residue_ss_exposure_heatmap_panav10seedens.png", dpi=300, bbox_inches='tight')
print('Saved -> Figs/residue_ss_exposure_heatmap_panav10seedens.(svg|png)')
plt.show()

print('\nExposure x SS marginal median |error| (ppm):')
print(df.groupby(['exposure', 'ss_class'])['abs_error'].median()
        .unstack('ss_class').reindex(columns=SS_ORDER).reindex(EXPOSURE_ORDER).round(3))

In [ ]:
# ================= same heatmap for every model + cross-model consistency =================
# Re-runs the cell-5 recipe (Pro / None / prediction-mask gate, raw |pred - gt|) on each model's
# own 10-seed-ensemble predictions. Residue rows use the SLAE-CS order so all heatmaps line up,
# and every model shares one colour scale.
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

FIG4 = "../../data/analysis/Fig4"
MODELS = {                       # file suffix -> label
    "":              "SLAE-CS",
    "_esm3":         "ESM3 (str + seq)",
    "_mace":         "MACE-OFF23",
    "_esm3_seqonly": "ESM3 (seq only)",
    "_mpnn_ft":      "ProteinMPNN",
}
MIN_CELL_N = 20                  # cells with fewer residues are left out of the consistency score


def build_df(pred_path):
    pred_data = json.load(open(pred_path))
    pred_dict, mask_dict = pred_data["mean"], pred_data["mask"]
    rows = []
    for entry in gt_data["entry_ID"].values():
        key = dictionary.get(entry)
        prr_entry = prr_data.get(entry)
        if key is None or entry not in pred_dict or prr_entry is None:
            continue
        gt_seq, gt_shifts = gt_data["sequence"][key], gt_data["N"][key]
        n = len(gt_seq)
        pred_arr, pred_mask = np.array(pred_dict[entry]), np.array(mask_dict[entry], dtype=bool)
        if len(pred_arr) < n or len(pred_mask) < n:
            continue
        pred_arr, pred_mask = pred_arr[:n], pred_mask[:n]
        prr_lookup = {r["resnum"]: r for r in prr_entry}
        mask = (np.array([r != "P" for r in gt_seq]) & np.array([g is not None for g in gt_shifts])
                & pred_mask)
        for idx in np.where(mask)[0]:
            rec = prr_lookup.get(int(idx) + 1)
            if rec is None or rec.get("ss_class") is None or rec.get("exposure") not in EXPOSURE_ORDER:
                continue
            rows.append({"entry": entry, "residue_type": gt_seq[idx], "ss_class": rec["ss_class"],
                         "exposure": rec["exposure"],
                         "abs_error": abs(float(pred_arr[idx]) - float(gt_shifts[idx]))})
    return pd.DataFrame(rows)


dfs = {lab: build_df(f"{FIG4}/results/predictions_N_seed_ensemble_10seed{tag}.json")
       for tag, lab in MODELS.items()}
for lab, d in dfs.items():
    print(f"{lab:18s} residues={len(d)}  proteins={d.entry.nunique()}")

# residue order (grouped, by SLAE-CS median) -- same rule as the SLAE-CS heatmap
AA_GROUPS = {
    "Aliphatic": ["G", "A", "V", "L", "I", "M"], "Aromatic": ["F", "Y", "W"],
    "Polar": ["S", "T", "C", "N", "Q"], "Positive": ["K", "R", "H"], "Negative": ["D", "E"],
}
res_med = dfs["SLAE-CS"].groupby("residue_type")["abs_error"].median()
group_sorted = {g: sorted([a for a in aas if a in res_med.index], key=lambda a: res_med[a], reverse=True)
                for g, aas in AA_GROUPS.items()}
group_sorted = {g: v for g, v in group_sorted.items() if v}
groups_ordered = sorted(group_sorted, key=lambda g: max(res_med[a] for a in group_sorted[g]), reverse=True)
row_order = [a for g in groups_ordered for a in group_sorted[g]]
group_bounds, acc = [], 0
for g in groups_ordered:
    group_bounds.append((g, acc, acc + len(group_sorted[g])))
    acc += len(group_sorted[g])

panels_by_model = {lab: {e: tuple(t.reindex(row_order) for t in pivots(d[d.exposure == e]))
                         for e in EXPOSURE_ORDER} for lab, d in dfs.items()}

# one shared colour scale across models: 5th-95th percentile of all cell medians
all_med = np.concatenate([panels_by_model[l][e][0].values.ravel() for l in MODELS.values() for e in EXPOSURE_ORDER])
VMIN, VMAX = np.nanpercentile(all_med, [5, 95])
print(f"shared colour scale: {VMIN:.2f} - {VMAX:.2f} ppm")

set_style(base=14)
ss_disp = ["Helix", "Sheet", "Coil"]
thresh = VMIN + 0.65 * (VMAX - VMIN)

In [ ]:
# ================= all five models side by side: one panel per model, buried | gap | exposed =================
# Uses panels_by_model / row_order / group_bounds / VMIN / VMAX from the cell above.
# Each panel is buried H E C, a blank spacer, then exposed H E C; residue rows follow the SLAE-CS
# order and are shared across panels; no per-cell annotation.
set_style(base=13)
labels = list(MODELS.values())
ss_short = ["H", "E", "C"]
n_ss = len(ss_short)
GAP = 0.3                                                   # spacer width, in heatmap-column units
blocks_x0 = [k * (n_ss + GAP) for k in range(len(EXPOSURE_ORDER))]   # left edge (column index) of each block

fig, axes = plt.subplots(1, len(labels), figsize=(2.3 * len(labels) + 1.4, 6.2),
                         sharey=True, gridspec_kw=dict(wspace=0.22))
for c, (ax, lab) in enumerate(zip(axes, labels)):
    for x0, e in zip(blocks_x0, EXPOSURE_ORDER):
        med = panels_by_model[lab][e][0]
        im = ax.imshow(med.values, aspect="auto", cmap="YlOrRd", vmin=VMIN, vmax=VMAX,
                       extent=(x0 - 0.5, x0 + n_ss - 0.5, len(row_order) - 0.5, -0.5))
        for _, start, _ in group_bounds[1:]:                 # residue-group separators, only across this block
            ax.hlines(start - 0.5, x0 - 0.5, x0 + n_ss - 0.5, color="k", lw=0.9)
        ax.add_patch(plt.Rectangle((x0 - 0.5, -0.5), n_ss, len(row_order),
                                   fill=False, ec="k", lw=1.0))   # frame around each block
        ax.text(x0 + (n_ss - 1) / 2, -0.8, e.capitalize(), ha="center", va="bottom", fontsize=11)
    ax.set_xlim(-0.5, blocks_x0[-1] + n_ss - 0.5)
    ax.set_ylim(len(row_order) - 0.5, -0.5)
    ax.set_xticks([x0 + j for x0 in blocks_x0 for j in range(n_ss)])
    ax.set_xticklabels(ss_short * len(EXPOSURE_ORDER), fontsize=11)
    ax.tick_params(length=0)
    for s in ax.spines.values():                             # the per-block frames replace the axes frame
        s.set_visible(False)
    ax.set_title(lab, fontsize=13, pad=20)
    if c == 0:
        ax.set_yticks(range(len(row_order)))
        ax.set_yticklabels(row_order, fontsize=12)
        for g, start, end in group_bounds:                   # residue-group labels left of the first panel
            ax.text(-1.4, (start + end - 1) / 2, g, ha="right", va="center", fontsize=11)

fig.supxlabel("Secondary structure (H = helix, E = sheet, C = loop)", fontsize=13, y=0.04)
cbar = fig.colorbar(im, ax=axes, fraction=0.02, pad=0.015)
cbar.set_label("Median |error| (ppm)", rotation=270, labelpad=14)
fig.savefig("Figs/residue_ss_exposure_heatmap_panav10seedens_allmodels_row.svg", bbox_inches="tight")
fig.savefig("Figs/residue_ss_exposure_heatmap_panav10seedens_allmodels_row.png", dpi=300, bbox_inches="tight")
print("saved residue_ss_exposure_heatmap_panav10seedens_allmodels_row")
plt.show()
